In [1]:
! pip install nashpy
! pip install open_spiel==1.6
#! pip install cvxopt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 5.8 MB/s eta 0:00:00


In [3]:
# New Approach

# Date Jul 02, 2025
# Zero sum model
# logic for debtoru_solver as per paper, uniform column probabilities for undominated strategies

import numpy as np
import pyspiel
from open_spiel.python.algorithms import lp_solver
import textwrap
import time

from itertools import product
from open_spiel.python.algorithms.lp_solver import LinearProgram
import cvxopt

def extract_row_payoff_matrix(game):
    return np.array(game.row_utilities())

def weakly_dominated_rows(matrix):
    keep = [1] * matrix.shape[0]
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[0]):
            if i == j:
                continue
            if np.all(matrix[i, :] <= matrix[j, :]) and np.any(matrix[i, :] < matrix[j, :]):
                keep[i] = 0
                break
    return keep

def weakly_dominated_columns(matrix):
    keep = [1] * matrix.shape[1]
    for j in range(matrix.shape[1]):
        for k in range(matrix.shape[1]):
            if j == k:
                continue
            if np.all(matrix[:, j] >= matrix[:, k]) and np.any(matrix[:, j] > matrix[:, k]):
                keep[j] = 0
                break
    return keep

def ieds_zero_sum_weak_flags(matrix):
    """
    Iterated Elimination of Weakly Dominated Strategies (IEDS) for 2-player zero-sum games.

    Args:
        matrix (np.ndarray): Row player's payoff matrix (m x n)

    Returns:
        reduced_matrix (np.ndarray): Matrix after IEDS
        row_flag (List[int]): 0 if row was eliminated, 1 if retained
        col_flag (List[int]): 0 if column was eliminated, 1 if retained
    """
    # print("\nMatrix input inside IEDS")
    # print(matrix)
    A = matrix.copy()
    orig_rows, orig_cols = matrix.shape
    row_flag = [1] * orig_rows
    col_flag = [1] * orig_cols

    active_rows = list(range(orig_rows))
    active_cols = list(range(orig_cols))

    changed = True
    while changed:
        changed = False

        # Eliminate weakly dominated rows
        current_matrix = A[np.ix_(active_rows, active_cols)]
        row_keep = weakly_dominated_rows(current_matrix)
        if sum(row_keep) < len(row_keep):
            new_active_rows = [r for r, keep in zip(active_rows, row_keep) if keep == 1]
            for r in active_rows:
                if r not in new_active_rows:
                    row_flag[r] = 0
            active_rows = new_active_rows
            changed = True

        # Eliminate weakly dominated columns
        current_matrix = A[np.ix_(active_rows, active_cols)]
        col_keep = weakly_dominated_columns(current_matrix)
        if sum(col_keep) < len(col_keep):
            new_active_cols = [c for c, keep in zip(active_cols, col_keep) if keep == 1]
            for c in active_cols:
                if c not in new_active_cols:
                    col_flag[c] = 0
            active_cols = new_active_cols
            changed = True

    reduced_matrix = A[np.ix_(active_rows, active_cols)]
    return reduced_matrix, row_flag, col_flag

def compute_column_player_value(p0_sol, p1_sol, payoff_matrix):
    """
    Computes the expected value of the game for the row player.

    Parameters:
        p0_sol (np.ndarray): Row player's strategy (size: [num_rows])
        p1_sol (np.ndarray): Column player's strategy (size: [num_cols])
        payoff_matrix (np.ndarray): Row player's payoff matrix (shape: [num_rows x num_cols])

    Returns:
        float: Expected value of the game for the row player
    """
    p0 = np.array(p0_sol).reshape(-1, 1)  # Column vector
    p1 = np.array(p1_sol).reshape(1, -1)  # Row vector
    return float(np.sum(p0 * payoff_matrix * p1))

def expand_strategy(short_strategy, flag):
    """
    Expands a strategy vector to full size using a flag vector.

    Parameters:
        short_strategy (List[float]): Strategy values for undominated strategies.
        flag (List[int]): Binary flag (1 = keep, 0 = dominated).

    Returns:
        np.ndarray: Full strategy with 0s in dominated positions.
    """
    full_strategy = np.zeros(len(flag))
    idx = 0
    for i, keep in enumerate(flag):
        if keep == 1:
            full_strategy[i] = short_strategy[idx]
            idx += 1
    return full_strategy

def solve_with_debtoru_solver(game, row_flag, dominance_mask):
    assert isinstance(game, pyspiel.MatrixGame)
    assert game.get_type().information == pyspiel.GameType.Information.ONE_SHOT
    assert game.get_type().utility == pyspiel.GameType.Utility.ZERO_SUM

    num_rows = game.num_rows()
    num_cols = game.num_cols()
    cvxopt.solvers.options["show_progress"] = False

    start_time1 = time.time()
    # ----------------------------
    # Row player's LP (OpenSpiel version)
    # ----------------------------
    lp0 = LinearProgram(lp_solver.OBJ_MAX)
    for r in range(num_rows):
        lp0.add_or_reuse_variable(r, lb=0)
    lp0.add_or_reuse_variable(num_rows)  # V
    lp0.set_obj_coeff(num_rows, 1.0)     # max V

    for c in range(num_cols):
        lp0.add_or_reuse_constraint(c, lp_solver.CONS_TYPE_GEQ)
        for r in range(num_rows):
            lp0.set_cons_coeff(c, r, game.player_utility(0, r, c))
        lp0.set_cons_coeff(c, num_rows, -1.0)  # -V >= 0

    lp0.add_or_reuse_constraint(num_cols + 1, lp_solver.CONS_TYPE_EQ)
    lp0.set_cons_rhs(num_cols + 1, 1.0)
    for r in range(num_rows):
        lp0.set_cons_coeff(num_cols + 1, r, 1.0)

    sol = lp0.solve()
    p0_sol = sol[:-1]      # row strategy
    p0_sol_val = sol[-1]   # game value for row player

    end_time1 = time.time()
    elapsed1 = end_time1 - start_time1
    num_undominated1 = sum(row_flag)
    print(f"\nExecution time of row: {elapsed1:.6f} seconds")
    print(f"\nNumber of undominated row: {num_undominated1}")

    start_time2 = time.time()
    # ----------------------------
    # Column strategy (modified)
    # ----------------------------
    matrix = extract_row_payoff_matrix(game)
    # reduced_matrix, row_flag, dominance_mask = ieds_zero_sum_weak_flags(matrix)
    # print("\nRow Mask\n",row_flag)
    # print("\nColumn Mask\n",dominance_mask)
    num_undominated = sum(dominance_mask)

    if num_undominated == 0:
        raise ValueError("All column strategies are dominated — no valid strategy remains.")

    p1_sol = np.zeros(num_cols)
    uniform_prob = 1.0 / num_undominated
    for i in range(num_cols):
        p1_sol[i] = uniform_prob

    # print("\np0_sol\n", p0_sol)
    # print("\np1_sol\n", p1_sol)
    # ----------------------------
    # Return as in original OpenSpiel
    # ----------------------------
    p1_sol_val = compute_column_player_value(p0_sol, p1_sol, matrix)  # skipped LP for column player

    end_time2 = time.time()
    elapsed2 = end_time2 - start_time2
    print(f"\nExecution time of column: {elapsed2:.6f} seconds")
    print(f"\nNumber of undominated columns: {num_undominated}")
    p0_sol_full = expand_strategy(p0_sol, row_flag)
    p1_sol_full = expand_strategy(p1_sol, dominance_mask )

    return p0_sol_full, p1_sol_full, p0_sol_val, p1_sol_val

def generate_strategies(num_agents, num_arsenals):
    """
    Generate all unique descending allocations (canonical form) of num_agents across num_arsenals.
    Each allocation is a tuple of non-negative integers that sum to num_agents,
    sorted in descending order. No repeated permutations.
    The result is sorted in reverse lexicographical order.
    """
    strategies = set()

    # Generate all possible combinations
    for combo in product(range(num_agents + 1), repeat=num_arsenals):
        if sum(combo) == num_agents:
            strategies.add(tuple(sorted(combo, reverse=True)))  # enforce canonical form

    return sorted(strategies, reverse=True)



#Compute and display the Nash equilibrium strategies and game value for the zero-sum game.

payoff_matrix = np.array([[1.0, 0.9189, 0.9139, 0.6035, 0.0  ],

                          [0.8194, 0.9185, 0.8883, 0.5526, 0.7833 ]












              ])

reduced_matrix, row_flag, dominance_mask = ieds_zero_sum_weak_flags(payoff_matrix)
print("\nreduced_matrix\n",reduced_matrix)
print("\nRow Mask\n",row_flag)
print("\nColumn Mask\n",dominance_mask)



# Create OpenSpiel matrix game
row_utilities = reduced_matrix.tolist()
col_utilities = [[-cell for cell in row] for row in row_utilities]  # zero-sum version
game = pyspiel.create_matrix_game(row_utilities, col_utilities)

start_time = time.time()
# Solve using LP solver
row_strategy, col_strategy, game_value_r, game_value_c = solve_with_debtoru_solver(game, row_flag, dominance_mask)
end_time = time.time()
elapsed = end_time - start_time

# Output strategies and game value
row_strategy_rounded = [round(p, 3) for p in row_strategy]
col_strategy_rounded = [round(p, 3) for p in col_strategy]
game_value_r_rounded = round(game_value_r, 3)
game_value_c_rounded = round(game_value_c, 3)

print("\nNash Equilibrium:")
formatted_row_strategy = [f"{float(x):.3f}" for x in row_strategy]
print("\nRow Player  Strategy:\n", textwrap.fill(str(formatted_row_strategy), width=100))

# print("\nColumn Player Strategy:\n", textwrap.fill(str(col_strategy_rounded), width=100))
formatted_col_strategy = [f"{float(x):.3f}" for x in col_strategy]
print("\nColumn Player Strategy:\n", textwrap.fill(str(formatted_col_strategy), width=100))

# print("\nValue of the Game (According to Row strategies of LP_Solver):". game_value_r)
print("\nValue of the Game (According to Row Prob from LP_Solver and uniform column prob):", game_value_c)
print(f"\nExecution time of debtoru_solver: {elapsed:.6f} seconds")


reduced_matrix
 [[0.6035 0.    ]
 [0.5526 0.7833]]

Row Mask
 [1, 1]

Column Mask
 [0, 0, 0, 1, 1]

Execution time of row: 0.002161 seconds

Number of undominated row: 2

Execution time of column: 0.000364 seconds

Number of undominated columns: 2

Nash Equilibrium:

Row Player  Strategy:
 ['0.277', '0.723']

Column Player Strategy:
 ['0.000', '0.000', '0.000', '0.500', '0.500']

Value of the Game (According to Row Prob from LP_Solver and uniform column prob): 0.5666765323434614

Execution time of debtoru_solver: 0.002775 seconds


In [2]:
23# Date Jul 02, 2025
# Zero sum model
#LP_Solver
import numpy as np
import pyspiel
from open_spiel.python.algorithms import lp_solver
import textwrap
import time

from itertools import product

def generate_strategies(num_agents, num_arsenals):
    """
    Generate all unique descending allocations (canonical form) of num_agents across num_arsenals.
    Each allocation is a tuple of non-negative integers that sum to num_agents,
    sorted in descending order. No repeated permutations.
    The result is sorted in reverse lexicographical order.
    """
    strategies = set()

    # Generate all possible combinations
    for combo in product(range(num_agents + 1), repeat=num_arsenals):
        if sum(combo) == num_agents:
            strategies.add(tuple(sorted(combo, reverse=True)))  # enforce canonical form

    return sorted(strategies, reverse=True)


#Compute and display the Nash equilibrium strategies and game value for the zero-sum game.

payoff_matrix = np.array([[1.0, 0.9189, 0.9139, 0.6035, 0.0  ],

                          [0.8194, 0.9185, 0.8883, 0.5526, 0.7833 ]





              ])

# Create OpenSpiel matrix game
row_utilities = payoff_matrix.tolist()
col_utilities = [[-cell for cell in row] for row in row_utilities]  # zero-sum version
game = pyspiel.create_matrix_game(row_utilities, col_utilities)

# Solve using LP solver
start_time = time.time()
row_strategy, col_strategy, game_value_r, game_value_c = lp_solver.solve_zero_sum_matrix_game(game)
end_time = time.time()
elapsed = end_time - start_time

# Output strategies and game value
row_strategy_rounded = [round(p, 3) for p in row_strategy]
col_strategy_rounded = [round(p, 3) for p in col_strategy]
game_value_r_rounded = round(game_value_r, 3)
game_value_c_rounded = round(game_value_c, 3)

print("\nNash Equilibrium:")
print("\nRow Player  Strategy:\n", textwrap.fill(str(row_strategy_rounded), width=100))
print("\nColumn Player  Strategy:\n", textwrap.fill(str(col_strategy_rounded), width=100))
print("\nValue of the Game (Expected  Payoff_Row):", game_value_r_rounded)
print("\nValue of the Game (Expected  Payoff_Col):", game_value_c_rounded)
print(f"Execution time of pure lp_solver: {elapsed:.6f} seconds")


Nash Equilibrium:

Row Player  Strategy:
 [0.277, 0.723]

Column Player  Strategy:
 [0.0, 0.0, 0.0, 0.939, 0.061]

Value of the Game (Expected  Payoff_Row): 0.567

Value of the Game (Expected  Payoff_Col): -0.567
Execution time of pure lp_solver: 0.062658 seconds
